# Notebook 4: Lockbox State, DLC Labeling Preparation

**Goal.** Set up everything needed to train a DLC model on the lockbox mechanisms, so that triangulating those keypoints (in a follow-up notebook) gives us scalar state per mechanism.

**What this notebook does**
1. Loads existing pipeline state (2D predictions, 3D mouse, calibration).
2. Defines a *lockbox state schema*: for each mechanism, which keypoints define it and what 1-DOF axis (or free 3D position) its state lives on, in the lockbox coordinate frame from Notebook 01 (calibration).
3. Checks per-camera observability of each mechanism's motion.
4. Selects ~150-250 frames to label using two complementary signals: (a) mouse-near-mechanism events and (b) uniform temporal coverage.
5. Creates a DLC project, sets the bodyparts to the lockbox keypoints, and extracts the selected frames into DLC's `labeled-data/` layout, one mixed project across all 3 views.

**Mechanism types supported by the schema**
- `revolute`, 1 DOF rotation about a fixed hinge axis (lever, knob).
- `prismatic`, 1 DOF translation along a fixed rail (slider).
- `free`, 3 DOF free translation in space (ball, removable piece). State is the triangulated 3D position itself.

## 0. Imports and paths

In [1]:
import json
import pickle
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- paths (edit if your layout differs) ---
DATA_ROOT = Path('../data').resolve()
SCENE = 'scene1'

# inputs from previous notebooks
EXPORT_2D   = DATA_ROOT / 'pipeline_export'     / SCENE / '2d_state.pkl'
PIPELINE_3D = DATA_ROOT / 'triangulate_render'  / SCENE / 'pipeline_state.pkl'
CALIB_TOML  = DATA_ROOT / 'lockbox_calibration' / SCENE / 'lockbox_calibration.toml'

# outputs from THIS notebook
OUT          = DATA_ROOT / 'lockbox_dlc' / SCENE
SCHEMA_PATH  = OUT / 'state_schema.json'
FRAMES_JSON  = OUT / 'selected_frames.json'
OUT.mkdir(parents=True, exist_ok=True)

print('out:', OUT)


out: /home/kenny/HTCV/data/lockbox_dlc/scene1


## 1. Load existing pipeline state

In [2]:
with open(EXPORT_2D, 'rb') as f:
    s2d = pickle.load(f)
with open(PIPELINE_3D, 'rb') as f:
    s3d = pickle.load(f)

points_3d        = s3d['points_3d']         # (T, n_kps, 3) mm, lockbox coords
valid_kps        = list(s3d['valid_kps'])
VIDEO_FOR_VIEW   = dict(s3d['VIDEO_FOR_VIEW'])
T, n_kps, _      = points_3d.shape

from aniposelib.cameras import CameraGroup
cgroup = CameraGroup.load(str(CALIB_TOML))
cam_names = [c.get_name() for c in cgroup.cameras]

print(f'frames: {T}, mouse keypoints: {n_kps}')
print(f'cameras: {cam_names}')
print(f'views available: {list(VIDEO_FOR_VIEW)}')


frames: 8190, mouse keypoints: 20
cameras: ['top', 'side', 'front']
views available: ['top', 'side', 'front']


## 2. Define the lockbox state schema

For each mechanism you fill in:

**Common fields**
- `name`, e.g. `lever1`, `slider1`, `ball1`.
- `type`, `'revolute'`, `'prismatic'`, or `'free'`.
- `state_keypoint`, the moving keypoint you'll label every frame.
- `reference_keypoints`, fixed helper points on the same part (pivot, rail ends, cradle). Optional but useful: they let you derive axes from labels rather than CAD, and give the model geometric context.

**For `revolute` / `prismatic`**
- `axis_origin_lockbox`, 3D point on the kinematic axis, in the calibration's world frame.
- `axis_direction_lockbox`, unit vector along the axis (hinge for revolute, rail for prismatic).
- `state_extraction`, `'angle_around_axis'` or `'project_on_axis'`.
- `range_rad` / `range_mm`, expected min/max state. Sanity-filter later.

**For `free`** (ball-like)
- `rest_position_lockbox`, where the part sits at rest (cradle/cup location). Used for frame selection. No kinematic axis.
- `state_extraction`, `'identity_3d'`. State = the triangulated 3D position.
- `range_mm_box`, optional bounding box for filtering wild triangulations.
- `enabled_by`, optional, documentation only: name of another mechanism that must change state first. The downstream NN learns this from data.

**Coordinate frame reminder.** All axes/origins must be in the same frame as your Notebook 01 `solvePnP` CAD points. If unsure, label a `reference_keypoint` at the pivot/origin position and read its triangulated coords off scene 1, that bootstraps the schema from data and sidesteps the CAD-frame question.

Below: the lever (tracking the push-end), a free ball, plus commented templates for sliders and horizontal rotators.

In [ ]:
LOCKBOX_SCHEMA = {
    'coord_frame': 'lockbox',
    'units': 'mm',
    'mechanisms': [
        # ===================================================================
        # LEVER 1 — L-shaped, 90° flip. Top-view-only extraction.
        # Angle is computed relative to `rest_direction_lockbox`:
        #   rest  -> 0
        #   flipped -> ~+pi/2 (magnitude. Sign depends on 2D geometry)
        # Engagement uses abs_above so sign doesn't matter.
        # ===================================================================
        {
            'name': 'lever1',
            'type': 'revolute',
            'dof': 1,
            'state_keypoint': 'lever1_push_end',
            'reference_keypoints': ['lever1_pivot'],
            'axis_origin_lockbox':    [4.32, 32.51, 27.39],
            'axis_direction_lockbox': [1.0, 0.0, 0.0],
            # (inferred from data: mostly +z, tiny +x).
            'rest_direction_lockbox': [0.14, 0.016, 0.99],
            'preferred_view': 'top',
            'state_extraction': 'angle_relative_top_view',
            # Wide range; the value is angle-from-rest, not absolute orientation.
            'range_rad': [-np.pi/2 - 0.2, np.pi/2 + 0.2],
            'notes': 'Top-view-only extraction: project static 3D pivot to top 2D, '
                     'decompose tracked 2D push-end on the projected rest/perp basis, '
                     'angle = atan2(perp, along_rest).'
        },

        # ===================================================================
        # SLIDER 1 — knob rest at x ≈ -43.78
        # ===================================================================
        {
            'name': 'slider1',
            'type': 'prismatic',
            'dof': 1,
            'state_keypoint': 'slider1_knob',
            'reference_keypoints': ['slider1_stabilizer'],
            # Origin moved to observed knob rest:
            'axis_origin_lockbox':    [-43.78, -22.03, 24.25],
            'axis_direction_lockbox': [-1.0, 0.0, 0.0],
            'state_extraction': 'project_on_axis',
            # Observed travel: p5 ≈ 0, p95 ≈ 30 mm. Old [0, 9.2] killed everything.
            'range_mm': [-2.0, 40.0],
        },

        # ===================================================================
        # BALL 1 — 3-DOF free. Explicit removal threshold from rest position.
        # ===================================================================
        {
            'name': 'ball1',
            'type': 'free',
            'dof': 3,
            'state_keypoint': 'ball1_center',
            'reference_keypoints': [],
            'rest_position_lockbox': [50.0, -22.57, 8.2],
            'removed_radius_mm': 30.0,
            'state_extraction': 'distance_from_rest',
            'range_mm_box': None,
            'notes': 'Distance from rest_position_lockbox. '
                     'engaged_mask = value > removed_radius_mm.'
        },

        # ===================================================================
        # COVER 1 — slides in -y. Origin moved to observed marker rest.
        # ===================================================================
        {
            'name': 'cover1',
            'type': 'prismatic',
            'dof': 1,
            'state_keypoint': 'cover_marker',
            'reference_keypoints': ['cover_guidance_left', 'cover_guidance_right'],
            'axis_origin_lockbox':    [48.27, 9.64, 25.80],   # observed rest
            'axis_direction_lockbox': [0.0, -1.0, 0.0],
            'state_extraction': 'project_on_axis',
            'range_mm': [-2.0, 30.0],
        },
    ],
}

# --- validate per-type ---
for m in LOCKBOX_SCHEMA['mechanisms']:
    t = m['type']
    if t in ('revolute', 'prismatic'):
        d = np.asarray(m['axis_direction_lockbox'], dtype=float)
        n = np.linalg.norm(d)
        assert n > 1e-6, f'{m["name"]}: zero-length axis direction'
        m['axis_direction_lockbox'] = (d / n).tolist()
        assert 'axis_origin_lockbox' in m, f'{m["name"]}: missing axis_origin_lockbox'
        if t == 'revolute' and 'rest_direction_lockbox' in m:
            r = np.asarray(m['rest_direction_lockbox'], float)
            nr = np.linalg.norm(r)
            assert nr > 1e-6, f'{m["name"]}: zero-length rest_direction'
            m['rest_direction_lockbox'] = (r / nr).tolist()
    elif t == 'free':
        assert 'rest_position_lockbox' in m, f'{m["name"]}: free needs rest_position_lockbox'
        assert 'removed_radius_mm'    in m, f'{m["name"]}: free needs removed_radius_mm'
    else:
        raise ValueError(f'{m["name"]}: unknown type {t!r}')

# --- collect bodyparts for DLC ---
all_lockbox_kps = []
for m in LOCKBOX_SCHEMA['mechanisms']:
    all_lockbox_kps.append(m['state_keypoint'])
    all_lockbox_kps.extend(m.get('reference_keypoints', []))
all_lockbox_kps = sorted(set(all_lockbox_kps))

print(f'mechanisms defined: {len(LOCKBOX_SCHEMA["mechanisms"])}')
for m in LOCKBOX_SCHEMA['mechanisms']:
    print(f'  {m["name"]:<10} ({m["type"]:<9} dof={m["dof"]})  -> {m["state_keypoint"]}')
print(f'\ntotal lockbox bodyparts ({len(all_lockbox_kps)}): {all_lockbox_kps}')

with open(SCHEMA_PATH, 'w') as f:
    json.dump(LOCKBOX_SCHEMA, f, indent=2, default=float)
print(f'saved -> {SCHEMA_PATH}')

mechanisms defined: 4
  lever1     (revolute  dof=1)  -> lever1_push_end
  slider1    (prismatic dof=1)  -> slider1_knob
  ball1      (free      dof=3)  -> ball1_center
  cover1     (prismatic dof=1)  -> cover_marker

total lockbox bodyparts (8): ['ball1_center', 'cover_guidance_left', 'cover_guidance_right', 'cover_marker', 'lever1_pivot', 'lever1_push_end', 'slider1_knob', 'slider1_stabilizer']
saved -> /home/kenny/HTCV/data/lockbox_dlc/scene1/state_schema.json


## 3. Per-camera motion visibility

For 1-DOF mechanisms: score how observable each mechanism's motion is in each camera (motion perpendicular to view direction to fully visible, motion along view direction to invisible).

For `free` mechanisms: motion direction isn't constrained, so every camera contributes to triangulation. We mark these as "all" rather than a single score.

In [4]:
def cam_view_dir_world(cam):
    """Camera's optical axis (+Z in camera frame) expressed in world frame."""
    rvec = np.asarray(cam.get_rotation())
    R, _ = cv2.Rodrigues(rvec)          # R maps world -> camera (x_cam = R @ x_world + t)
    return R[2]                          # 3rd row = camera +Z expressed in world coords

def motion_visibility(motion_dir_world, cam):
    v = cam_view_dir_world(cam)
    return float(1.0 - abs(np.dot(motion_dir_world, v)))

def representative_motion_dirs(mech):
    """Motion-direction unit vectors to score. None for 'free'."""
    if mech['type'] == 'prismatic':
        return [np.asarray(mech['axis_direction_lockbox'])]
    if mech['type'] == 'revolute':
        hinge = np.asarray(mech['axis_direction_lockbox'])
        tmp = np.array([1.0, 0.0, 0.0]) if abs(hinge[0]) < 0.9 else np.array([0.0, 1.0, 0.0])
        p1 = np.cross(hinge, tmp); p1 /= np.linalg.norm(p1)
        p2 = np.cross(hinge, p1)
        return [p1, p2]
    return None  # free

rows = []
for m in LOCKBOX_SCHEMA['mechanisms']:
    dirs = representative_motion_dirs(m)
    for cam in cgroup.cameras:
        if dirs is None:
            rows.append({'mechanism': m['name'], 'camera': cam.get_name(), 'visibility': 'all'})
        else:
            score = min(motion_visibility(d, cam) for d in dirs)
            rows.append({'mechanism': m['name'], 'camera': cam.get_name(), 'visibility': f'{score:.3f}'})
vis = pd.DataFrame(rows).pivot(index='mechanism', columns='camera', values='visibility')
print('Motion visibility per camera (higher = state changes more visible; "all" = free-3D, every cam contributes):')
print(vis)


Motion visibility per camera (higher = state changes more visible; "all" = free-3D, every cam contributes):
camera     front   side    top
mechanism                     
ball1        all    all    all
cover1     0.012  0.866  0.870
lever1     0.012  0.863  0.009
slider1    0.979  0.019  0.988


## 4. Frame selection

In [ ]:
SCENES = ['scene1']      # every mouse/session to represent

NEAR_RADIUS_MM   = 60
INTERACTION_KPS  = ['nose', 'left_front_paw', 'right_front_paw']
EVENT_GAP        = 15     # merge contacts < this many frames apart into one event
EVENT_STRIDE     = 3      # dense sampling inside an interaction event
EDGE_PAD         = 8      # frames kept either side of an event (motion onset/offset)
PER_EVENT_CAP    = 25     # ceiling per event so one long grab can't dominate
N_UNIFORM        = 40     # per-scene background coverage (rest appearance, lighting)
N_FRAMES_TARGET  = 350    # total across all scenes

# fail loudly on the duplicate-name bug instead of silently merging mechanisms
_names = [m['name'] for m in LOCKBOX_SCHEMA['mechanisms']]
assert len(_names) == len(set(_names)), f'duplicate mechanism names: {_names}'

def mech_proxy_position(m):
    return np.asarray(m['rest_position_lockbox'] if m['type'] == 'free'
                      else m['axis_origin_lockbox'], dtype=float)

def contiguous_events(frames, gap):
    frames = list(frames)
    if not frames:
        return []
    events, start, prev = [], frames[0], frames[0]
    for f in frames[1:]:
        if f - prev > gap:
            events.append((start, prev)); start = f
        prev = f
    events.append((start, prev))
    return events

def select_for_scene(points_3d, valid_kps, T):
    kp_idx   = {k: i for i, k in enumerate(valid_kps)}
    used_kps = [k for k in INTERACTION_KPS if k in kp_idx]
    if not used_kps:
        raise ValueError(f'none of {INTERACTION_KPS} in valid_kps={valid_kps}')
    mouse_pts = points_3d[:, [kp_idx[k] for k in used_kps], :]   # (T, n_used, 3)
    inter, counts = set(), {}
    for m in LOCKBOX_SCHEMA['mechanisms']:
        origin = mech_proxy_position(m)
        min_d  = np.nanmin(np.linalg.norm(mouse_pts - origin[None, None, :], axis=-1), axis=1)
        near   = np.where(min_d < NEAR_RADIUS_MM)[0]
        sampled = []
        for a, b in contiguous_events(near, EVENT_GAP):
            lo, hi = max(0, a - EDGE_PAD), min(T - 1, b + EDGE_PAD)
            s = list(range(lo, hi + 1, EVENT_STRIDE))
            if len(s) > PER_EVENT_CAP:
                s = [s[i] for i in np.linspace(0, len(s) - 1, PER_EVENT_CAP, dtype=int)]
            sampled += s
        inter.update(sampled)
        counts[m['name']] = (len(near), len(sampled))
    uniform  = np.linspace(0, T - 1, N_UNIFORM, dtype=int).tolist()
    selected = sorted(set(uniform) | inter)
    return selected, uniform, counts, used_kps

selection = {}        # scene -> {'frames','uniform','videos','T'}
for sc in SCENES:
    with open(DATA_ROOT / 'triangulate_render' / sc / 'pipeline_state.pkl', 'rb') as f:
        st = pickle.load(f)
    p3d, vkps, vids, Tsc = (st['points_3d'], list(st['valid_kps']),
                            dict(st['VIDEO_FOR_VIEW']), st['points_3d'].shape[0])
    frames, uniform, counts, used = select_for_scene(p3d, vkps, Tsc)
    selection[sc] = {'frames': frames, 'uniform': uniform, 'videos': vids, 'T': Tsc}
    print(f'[{sc}]  T={Tsc}  interaction kps={used}')
    for n, c in counts.items():
        print(f'      {n:<10} near={c[0]:>5}  sampled={c[1]:>4}')
    print(f'      uniform={len(uniform)}  scene total={len(frames)}')

# cap total across scenes, proportionally
total = sum(len(v['frames']) for v in selection.values())
if total > N_FRAMES_TARGET:
    for v in selection.values():
        keep = max(1, round(N_FRAMES_TARGET * len(v['frames']) / total))
        idx  = np.linspace(0, len(v['frames']) - 1, min(keep, len(v['frames'])), dtype=int)
        v['frames'] = [v['frames'][i] for i in idx]
total = sum(len(v['frames']) for v in selection.values())
print(f'\nTOTAL across {len(SCENES)} scenes: {total}')

with open(FRAMES_JSON, 'w') as f:
    json.dump({'scenes': {sc: {'frame_indices': list(map(int, v['frames']))}
                          for sc, v in selection.items()},
               'params': {'near_radius_mm': NEAR_RADIUS_MM, 'interaction_kps': INTERACTION_KPS,
                          'event_gap': EVENT_GAP, 'event_stride': EVENT_STRIDE,
                          'edge_pad': EDGE_PAD, 'per_event_cap': PER_EVENT_CAP,
                          'n_uniform': N_UNIFORM, 'n_target': N_FRAMES_TARGET}}, f, indent=2)
print(f'saved -> {FRAMES_JSON}')

# timeline per scene
fig, axes = plt.subplots(len(SCENES), 1, figsize=(12, 1.5 * len(SCENES)), squeeze=False)
for ax, sc in zip(axes[:, 0], SCENES):
    v = selection[sc]; uni = set(v['uniform'])
    ax.vlines([f for f in v['frames'] if f in uni], 0.6, 1.0, color='steelblue', lw=0.8)
    ax.vlines([f for f in v['frames'] if f not in uni], 0.0, 0.4, color='crimson', lw=0.8)
    ax.set_xlim(0, v['T']); ax.set_yticks([])
    ax.set_ylabel(sc, rotation=0, ha='right', va='center')
axes[-1, 0].set_xlabel('frame')
fig.suptitle('selected frames — blue=uniform, red=interaction', y=1.02)
plt.tight_layout(); plt.show()

[scene1]  T=8190  interaction kps=['nose']
      lever1     near= 4435  sampled= 812
      slider1    near=  765  sampled= 338
      ball1      near= 3923  sampled= 686
      cover1     near= 5433  sampled= 503
      uniform=40  scene total=2070

TOTAL across 1 scenes: 350
saved -> /home/kenny/HTCV/data/lockbox_dlc/scene1/selected_frames.json


/tmp/ipykernel_1168/246918649.py:47: RuntimeWarning: All-NaN slice encountered
  min_d  = np.nanmin(np.linalg.norm(mouse_pts - origin[None, None, :], axis=-1), axis=1)
/tmp/ipykernel_1168/246918649.py:104: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## DLC project (pooled) + config patch

In [ ]:
import deeplabcut as dlc
import yaml

EXPERIMENTER = 'htcv'
PERVIEW_ROOT = OUT / 'perview'
PERVIEW_ROOT.mkdir(parents=True, exist_ok=True)

# all lockbox bodyparts. Each view labels only what it sees, and Notebook 5
# prunes each project's bodyparts to its labeled subset before training.
bodyparts = sorted({m['state_keypoint'] for m in LOCKBOX_SCHEMA['mechanisms']} |
                   {k for m in LOCKBOX_SCHEMA['mechanisms'] for k in m.get('reference_keypoints', [])})

# view -> [(scene, video_path), ...]  (pools scenes within a per-view model)
VIEW_VIDEOS = {}
for sc, v in selection.items():
    for view, vp in v['videos'].items():
        VIEW_VIDEOS.setdefault(view, []).append((sc, Path(vp)))

def find_project(view):
    hits = sorted(PERVIEW_ROOT.glob(f'lockbox_{view}-{EXPERIMENTER}-*/config.yaml'))
    return hits[-1] if hits else None

PERVIEW_CONFIG = {}
for view, items in VIEW_VIDEOS.items():
    cfgp = find_project(view)
    if cfgp is None:                                  # create fresh
        cfgp = Path(dlc.create_new_project(
            f'lockbox_{view}', EXPERIMENTER, [str(p) for _, p in items],
            working_directory=str(PERVIEW_ROOT), copy_videos=False, multianimal=False))
        cfg = yaml.safe_load(open(cfgp))
        cfg['bodyparts'] = bodyparts; cfg['skeleton'] = []; cfg['numframes2pick'] = 0
        yaml.safe_dump(cfg, open(cfgp, 'w'), sort_keys=False)
        print(f'{view:<6} created {cfgp.parent.name}')
    else:                                             # reuse — don't clobber edits
        print(f'{view:<6} reuse   {cfgp.parent.name}')
    PERVIEW_CONFIG[view] = cfgp

Loading DLC 3.0.0rc13...
DLC loaded in light mode; you cannot use any GUI (labeling, relabeling and standalone GUI)


/home/kenny/HTCV/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


top    reuse   lockbox_top-htcv-2026-05-25
side   reuse   lockbox_side-htcv-2026-05-25
front  reuse   lockbox_front-htcv-2026-05-25


## 5. Extract selected frames into labeled-data/<stem>/

In [7]:
for view, items in VIEW_VIDEOS.items():
    proj = PERVIEW_CONFIG[view].parent
    for sc, vp in items:
        target = proj / 'labeled-data' / vp.stem
        target.mkdir(parents=True, exist_ok=True)
        frames = selection[sc]['frames']
        if len(list(target.glob('*.png'))) >= len(frames):
            print(f'{view:<6} {sc} already extracted — skip'); continue
        cap, nw, nf = cv2.VideoCapture(str(vp)), 0, 0
        for fi in frames:
            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
            ok, frame = cap.read()
            if not ok: nf += 1; continue
            cv2.imwrite(str(target / f'img{fi:08d}.png'), frame); nw += 1
        cap.release()
        print(f'{view:<6} {sc} -> {target.name} (wrote {nw}, failed {nf})')

top    scene1 already extracted — skip
side   scene1 already extracted — skip
front  scene1 already extracted — skip


## 6. Next steps, seed, train, then refine